# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 4: Prompt Engineering</font>

# <font color="#003660">Notebook 3: Prompting with the OpenAI-Compatible API</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know why the OpenAI-compatible API is the base for almost everything you will build with LLMs, <br>
        ... can get an API key for the university's AI-Gateway, keep it in a <code>.env</code> file, and connect to it, <br>
        ... can send messages, system prompts, and few-shot examples with <code>chat.completions</code>, <br>
        ... can control the generation with <code>temperature</code>, <code>seed</code>, <code>max_tokens</code>, and <code>stop</code>.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [Raschka (2024): Building a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)
* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

Documentation for this notebook:

* [AI-Gateway of Paderborn University](https://hilfe.uni-paderborn.de/AI-Gateway)
* [OpenAI Python SDK: chat completions](https://platform.openai.com/docs/api-reference/chat/create?lang=python)
* OpenAI-compatible endpoints of other providers: [Ollama](https://docs.ollama.com/api/openai-compatibility), [Hugging Face Inference Providers](https://huggingface.co/docs/inference-providers/index?python-clients=openai&javascript-clients=openai), [vLLM](https://docs.vllm.ai/)

## Where We Are

* **4.01 and 4.02:** you wrote prompt templates by hand and used Ollama's own API.
* **Now:** the API you will use in your projects, the **OpenAI-compatible API**.

There are no new prompting concepts here. You do the same things as before, just through a different API.

## Why This API?

**In short: learn it once, use it everywhere.**

* OpenAI designed this interface for its own models. It became a de facto standard.
* **Every provider speaks it:** OpenAI, Ollama, vLLM, Hugging Face, cloud providers, and even our university.
* **Everything that comes later builds on it:** structured output, tool calling, RAG, agents. They are all requests to `chat.completions`.
* **Switching the provider means changing three things:** the address (`base_url`), the key (`api_key`), and the model name.

Using the OpenAI API *format* does not mean using OpenAI's *models*. In this notebook, we use the models of the university's **AI-Gateway**.

## Setup: The AI-Gateway of Paderborn University

The **AI-Gateway** gives you access to large language models with your university account. You need two things: access (once) and an API key.

### Step 1: Request access (only once)

1. Open the [Service-Portal](https://sp.upb.de) and log in with your university account.
2. Go to **Benutzerverwaltung** → **Weitere Dienste** → tab **Lehre und Studium**.
3. Find **KI-Chat-Oberfläche**, click **Optionen** → **Beantragen**. Access is granted automatically.

The full instructions, including the terms of use: [hilfe.uni-paderborn.de/AI-Gateway#Beantragen](https://hilfe.uni-paderborn.de/AI-Gateway#Beantragen)

After that, you can log in with your university account on two pages:

| Page | What it is for |
|---|---|
| [ai-gateway.upb.de](https://ai-gateway.upb.de) | API keys, for using the models from code (this notebook) |
| [ai-chat.upb.de](https://ai-chat.upb.de) | chatting with the models in the browser |

### Step 2: Create an API key

1. Connect to the university network or the VPN.
2. Open [ai-gateway.upb.de](https://ai-gateway.upb.de) and log in.
3. Click **API-Key erstellen** and copy the key.
4. In the repo folder (the one that contains `Session_04`, `Session_05`, ...), copy the file `.env.example` and name the copy `.env`.
5. Open `.env` and paste your key behind `AI_GATEWAY_KEY=` (no quotes, no spaces).

You do this once. All notebooks of the course read the key from this one file.

Treat the key like a password. Keys expire after a while; then you simply create a new one.

**Run this notebook on your own computer, connected to the university network or the VPN.** Google Colab runs outside the university network and cannot reach the gateway.

### Step 3: Connect from Python

We install the `openai` package.

In [ ]:
%pip install -q openai python-dotenv

The next cell loads your key with `load_dotenv()`. It looks for `.env` in the notebook's folder and then in the folders above, so it finds the file in the repo folder.

Look at the client: it is the normal OpenAI client. We only point it to the gateway with `base_url`. The gateway's address also comes from `.env` (`AI_GATEWAY_BASE`).

In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file: searches this folder first, then the folders above (the repo folder)
API_KEY = os.getenv("AI_GATEWAY_KEY")
AI_GATEWAY_BASE = os.getenv("AI_GATEWAY_BASE")  # address of the university's AI-Gateway
if not API_KEY or not AI_GATEWAY_BASE:
    raise ValueError("AI_GATEWAY_KEY or AI_GATEWAY_BASE missing. Copy .env.example to .env in the repo folder and paste your key there.")

client = OpenAI(
    base_url=AI_GATEWAY_BASE,  # the university's AI-Gateway instead of OpenAI's servers
    api_key=API_KEY,
)

**Never write the key into the notebook itself.** Notebooks end up on GitHub; the `.env` file does not, because it is listed in `.gitignore`.

### Step 4: Check the connection

The gateway lists the models you can use. An error here? Check the VPN and the key.

In [ ]:
for model in client.models.list():
    print(model.id)

We use **Qwen3.8** with 27 billion parameters. The `-no-think` at the end of the name means that thinking (4.02, Part 3) is switched off: the model answers right away. The list depends on your account, and you can try other models later.

In [ ]:
MODEL = "hs-de.qwen3.8-27b-no-think"  # Qwen3.8 (27B) with thinking switched off; you can try any other id from the list

### One small helper

`score()` works like in 4.01 and 4.02, but with messages instead of a raw template:

* It fills the placeholder `{text}` in every message with each example.
* It sends the messages with `client.chat.completions.create`.
* It counts the correct answers.

**Good to know:** `score()` sends one request per example. Some models on the gateway have a limit of requests per minute. If a cell pauses, the client is waiting and then continues by itself.

In [ ]:
import re

GREEN, ORANGE, RESET = "\033[92m", "\033[33m", "\033[0m"


def check(answer, expected):
    """Looks for the expected answer behind "The answer is" or, if the model did not write that, in the first line."""
    answer = answer.split("</think>")[-1]  # ignore the reasoning of reasoning models
    position = answer.lower().find("answer is")
    if position >= 0:
        answer = answer[position + len("answer is"):]
    first_line = next((line.strip() for line in answer.splitlines() if line.strip()), "")
    correct = re.search(rf"\b{re.escape(expected)}\b", first_line, re.IGNORECASE) is not None
    return correct, first_line


def fill(messages, text):
    """Replaces the placeholder {text} in every message."""
    return [{**message, "content": message["content"].replace("{text}", text)} for message in messages]


def score(messages, examples, model=None, temperature=0, max_tokens=50, **params):
    """Sends the messages for every example and counts the correct answers."""
    correct, prompt_tokens, tokens = 0, 0, 0
    for text, expected in examples:
        response = client.chat.completions.create(
            model=model or MODEL, messages=fill(messages, text), temperature=temperature, max_tokens=max_tokens, **params
        )
        ok, answer = check(response.choices[0].message.content or "", expected)
        correct += ok
        prompt_tokens += response.usage.prompt_tokens
        tokens += response.usage.completion_tokens
        mark = f"{GREEN}✓{RESET}" if ok else f"{ORANGE}✗{RESET}"
        print(f"{mark} expected: {expected:<10} model: {answer[:70]}")
    print(f"\n{correct}/{len(examples)} correct, {prompt_tokens} prompt tokens, {tokens} tokens generated")

## Part 1: One Request

You know the messages format from 4.02. The OpenAI-compatible API takes exactly the same list:

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
print(response.choices[0].message.content)

The response is a JSON object. Besides the answer, look for two things:

* `finish_reason`: why the generation stopped
* `usage`: how many tokens were used

In [ ]:
print(response.model_dump_json(indent=4))

Behind the scenes, the server does what you did by hand in 4.02: it puts your messages into the model's **chat template**. The API is just a more convenient way to get to the same template.

## Part 2: System Prompts and Few-Shot Examples as Messages

**In short: system prompts and few-shot examples are just messages.**

* **System prompt:** the first message in the list.
* **Few-shot examples:** earlier `user` and `assistant` messages that you write yourself. The model sees them as earlier turns of the conversation.

Let's do the sentiment task from 4.01:

In [ ]:
# ten short movie reviews with their sentiment
REVIEWS = [
    ("A wonderful film with brilliant actors.", "positive"),
    ("The plot was boring and far too long.", "negative"),
    ("I laughed from start to finish.", "positive"),
    ("A complete waste of two hours.", "negative"),
    ("Not bad at all, I would watch it again.", "positive"),
    ("The ending made no sense, and the dialogue was awful.", "negative"),
    ("Beautiful music and a touching story.", "positive"),
    ("I fell asleep halfway through.", "negative"),
    ("It was not as good as everyone said.", "negative"),
    ("My kids loved it, and so did I.", "positive"),
]

In [ ]:
zero_shot = [
    {"role": "system", "content": "Classify the sentiment of the movie review. Answer with positive or negative only."},
    {"role": "user", "content": "{text}"},
]
score(zero_shot, REVIEWS)

### Your turn

1. Add one or more examples as earlier `user` and `assistant` messages. Use your own reviews, not reviews from the list.
2. Compare the score with your best base-model template from 4.01. Does this model need the examples?

In [ ]:
# TODO: add your examples as earlier user and assistant messages
few_shot = [
    {"role": "system", "content": "Classify the sentiment of the movie review. Answer with positive or negative only."},
    {"role": "user", "content": "The acting was superb and the story kept me hooked."},
    {"role": "assistant", "content": "positive"},
    {"role": "user", "content": "Not as funny as the trailer promised."},
    {"role": "assistant", "content": "negative"},
    {"role": "user", "content": "{text}"},
]
score(few_shot, REVIEWS)

# Let's look at the outcome: compare the score and the prompt tokens with the zero-shot result above.

## Part 3: Controlling the Generation

**In short: the decoding parameters from Session 3 are part of every request.**

| Parameter | What it does |
|---|---|
| `temperature` | flattens (> 1) or sharpens (< 1) the next-token distribution; `0` means greedy decoding |
| `top_p` | samples only from the most likely tokens that together reach this probability |
| `seed` | fixes the random numbers, so that sampling gives the same result again |
| `max_tokens` | the maximum number of generated tokens |
| `stop` | a list of stop sequences, as in 4.01 |

`finish_reason` tells you why the generation stopped: `length` (maximum number of tokens) or `stop` (end-of-turn token or stop sequence).

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short poem about the sea."},
]

for seed in [1, 2, 3]:
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=1.0, seed=seed, max_tokens=60)
    print(f"seed {seed}, finish_reason: {response.choices[0].finish_reason}")
    print(response.choices[0].message.content, end="\n\n")

### Your turn

**Predict** before you run it:

* What changes with `temperature=0`?
* What is the `finish_reason` with `max_tokens=200`?
* And with `stop=["\n"]`?

In [ ]:
# TODO: change temperature, max_tokens, and stop, and compare the outputs and the finish_reason

# 1. temperature=0: greedy decoding. Do the three seeds still give different poems?
for seed in [1, 2, 3]:
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0, seed=seed, max_tokens=60)
    print(f"seed {seed}, finish_reason: {response.choices[0].finish_reason}")
    print(response.choices[0].message.content, end="\n\n")

# 2. max_tokens=200: more room. Which finish_reason do you get now?
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=1.0, seed=1, max_tokens=200)
print("finish_reason:", response.choices[0].finish_reason)
print(response.choices[0].message.content, end="\n\n")

# 3. stop=["\n"]: a line break is a stop sequence. Where does the poem end, and which finish_reason do you get?
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=1.0, seed=1, max_tokens=60, stop=["\n"])
print("finish_reason:", response.choices[0].finish_reason)
print(response.choices[0].message.content)

Not every provider supports every parameter, for example `seed`. If a parameter is ignored or rejected, check the provider's documentation.

## Part 4: Switching the Provider

**In short: change `base_url`, `api_key`, and the model name. The rest of your code stays.**

Some examples (not to run now):

```python
import os

# Ollama on your own computer, as in 4.01 and 4.02 (Ollama does not check the key, but the package needs one)
client = OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")
response = client.chat.completions.create(model="llama3.2:1b", messages=messages)

# a cloud provider, for example Hugging Face Inference Providers (needs a Hugging Face token)
client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])

# a vLLM server that you started yourself
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# OpenAI itself: the key is read from the environment variable OPENAI_API_KEY
client = OpenAI()
```

Three things still differ between providers:

* **The template.** Every model family has its own, and the provider applies it for you.
* **The parameters.** Not every provider supports every parameter, for example `seed`.
* **Thinking.** Some providers switch it on and off with the parameter `reasoning_effort`. Our gateway uses the model name instead: `gwdg.qwen3.8-27b` thinks, `gwdg.qwen3.8-27b-no-think` does not. The reasoning comes back in an extra field, `reasoning_content`.

**Never write an API key into a notebook.** Read it from the `.env` file, as in the setup above, or from an environment variable.

## Questions?

Anything unclear about the OpenAI-compatible API, the AI-Gateway, or the parameters? Ask now.

## What You Learned

* The **OpenAI-compatible API** is a de facto standard. Structured output, tool calling, RAG, and agents all build on it.
* The same code works with the university's **AI-Gateway**, Ollama, vLLM, Hugging Face, OpenAI, and many others. You switch with `base_url`, `api_key`, and the model name.
* `client.chat.completions.create` takes a **list of messages**. The provider turns it into the model's **chat template**.
* System prompts and few-shot examples are just messages: the system message comes first, and examples are earlier `user` and `assistant` turns.
* `temperature`, `top_p`, `seed`, `max_tokens`, and `stop` control the generation, and `finish_reason` tells you why it stopped.

Next: **4.04 - Prompt Engineering Best Practices**. A few habits for writing prompts and checking them.